In [1]:
# ============================================================
# REVIEWER 1 / PROBLEM 2
# E5-LARGE-INSTRUCT STRICT-FAIR FINE-TUNING — ONE CELL / ONE SCRIPT
# ============================================================
# Scientific design
# ------------------------------------------------------------
# Master paired set: exact CLEAN(question) intersection only
# Expected N = 6,759
# Split: seed=42, 90/10 -> train=6,083, test=676
#
# BASE FT:
#   clean question -> CLEAN baseline answer
# SEG FT:
#   morph-marked segmented question -> SAME CLEAN baseline answer
#
# Thus the textual answer-reference space and the underlying items are
# identical across conditions; only the question representation differs.
# Both FT models start independently from the SAME pretrained multilingual-E5-large-instruct checkpoint.
#
# Training:
#   CachedMultipleNegativesRankingLoss (MNRL objective), 5 epochs,
#   effective batch=32, cache mini-batch=2, lr=2e-5, warmup=10%,
#   same seeded batch order. PagedAdamW8bit is used only as a
#   memory-efficient optimizer implementation for E5-large-instruct on T4.
#   E5 query instruction formatting is kept identical across BASE and SEG.
#
# Evaluation:
#   Exact@1, TokenF1@1, MeanCos@1(QSim),
#   Semantic@1(answer cosine >= 0.85), BERTScoreF1@1
#
# Primary paired inference for FT BASE vs FT SEG:
#   continuous -> paired sign-flip permutation, 10,000
#   binary     -> exact two-sided McNemar
#   all        -> paired bootstrap 95% CI, 10,000
#   Holm correction across 5 primary metrics
#
# Optional non-FT reference is also evaluated on the exact same 676 items
# so FT results can be compared directly with the non-fine-tuned setting.
# ============================================================

import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import sys, re, json, glob, math, csv, time, random, hashlib, shutil, subprocess, zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

# ---------------------------
# 0. Dependencies
# ---------------------------
def ensure(pkg: str, import_name: Optional[str] = None):
    name = import_name or pkg.replace("-", "_")
    try:
        __import__(name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("numpy")
ensure("pandas")
ensure("scipy")
ensure("scikit-learn", "sklearn")
ensure("sentence-transformers", "sentence_transformers")
ensure("bert-score", "bert_score")
ensure("bitsandbytes", "bitsandbytes")

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses
import bitsandbytes as bnb
from bert_score import score as bert_score

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

# ---------------------------
# 1. CONFIG
# ---------------------------
BASE_PATH = "baseline_15000.json"
SEG_PATH  = "kazakh_segmented_15000.json"

MODEL_NAME = "intfloat/multilingual-e5-large-instruct"

SEED      = 42
TEST_SIZE = 0.10
SEM_THR   = 0.85
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

FINETUNE_EPOCHS    = 5
FINETUNE_BATCH      = 32   # effective MNRL batch, same as MiniLM
FINETUNE_MINI_BATCH = 2    # memory-saving cache chunk; does not change effective batch
FINETUNE_LR        = 2e-5
FINETUNE_WARMUP_FR = 0.10

# Complete comparison to the existing non-FT strict-fair result.
RUN_NONFT_REFERENCE = True
RUN_BERTSCORE       = True
BERTSCORE_BATCH     = 16 if torch.cuda.is_available() else 4
ENCODE_BATCH        = 32 if torch.cuda.is_available() else 4

N_BOOT     = 10_000
N_PERM     = 10_000
STAT_ALPHA = 0.05
STAT_SEED  = 20260901

# Unique directories prevent accidental reuse of an older, non-strict-fair FT model.
BASE_SAVE_DIR = "e5_large_strictfair_ft_BASE_qclean_aclean_seed42_5ep"
SEG_SAVE_DIR  = "e5_large_strictfair_ft_SEG_qmorph_aclean_seed42_5ep"
REUSE_SAVED_MODELS = True

OUT_DIR = Path("e5_large_strictfair_problem2_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError(
        "E5-LARGE-INSTRUCT full fine-tuning is configured for a CUDA GPU (T4 or better). "
        "In Colab select Runtime -> Change runtime type -> T4 GPU."
    )

# ---------------------------
# 2. Reproducibility
# ---------------------------
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)

print("=" * 72)
print("E5-LARGE-INSTRUCT STRICT-FAIR FINE-TUNING — REVIEWER 1 / PROBLEM 2")
print("=" * 72)
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Model: {MODEL_NAME}")
print("E5 input formatting: instruction-wrapped QUERY side; plain PASSAGE/answer side")
print(f"Effective FT batch={FINETUNE_BATCH}; CachedMNRL mini-batch={FINETUNE_MINI_BATCH}")

# ---------------------------
# 3. Find/load data
# ---------------------------
def find_data_path(filename: str) -> str:
    candidates = [
        filename,
        f"/content/{filename}",
        f"/mnt/data/{filename}",
    ]
    for p in candidates:
        if Path(p).is_file():
            return str(Path(p).resolve())
    hits = glob.glob(f"/content/**/{filename}", recursive=True) + glob.glob(f"./**/{filename}", recursive=True)
    hits = [h for h in hits if Path(h).is_file()]
    if hits:
        return str(Path(hits[0]).resolve())
    return ""

base_path = find_data_path(BASE_PATH)
seg_path  = find_data_path(SEG_PATH)

if (not base_path or not seg_path) and IN_COLAB:
    print("\nUpload BOTH files: baseline_15000.json and kazakh_segmented_15000.json")
    files.upload()
    base_path = find_data_path(BASE_PATH)
    seg_path  = find_data_path(SEG_PATH)

if not base_path or not seg_path:
    raise FileNotFoundError("Could not find baseline_15000.json and/or kazakh_segmented_15000.json")


def _normalize_records(data: Any) -> List[Dict[str, str]]:
    if not isinstance(data, list):
        raise ValueError("Parsed QA data must be a list.")
    out = []
    for x in data:
        if not isinstance(x, dict):
            continue
        q = x.get("question") or x.get("instruction") or ""
        a = x.get("answer") or x.get("response") or ""
        q, a = str(q).strip(), str(a).strip()
        if q and a:
            out.append({"question": q, "answer": a})
    if not out:
        raise ValueError("No valid question/answer records found.")
    return out


def load_qa_records(path: str) -> List[Dict[str, str]]:
    text = Path(path).read_text(encoding="utf-8", errors="ignore").strip()
    if not text:
        raise ValueError(f"Empty file: {path}")

    # Standard JSON array
    if text.startswith("["):
        try:
            return _normalize_records(json.loads(text))
        except Exception:
            pass

    # JSONL
    lines = [x.strip().rstrip(",") for x in text.splitlines() if x.strip()]
    if lines and lines[0].startswith("{"):
        try:
            return _normalize_records([json.loads(x) for x in lines])
        except Exception:
            pass

    # Robust brace scan fallback
    objs, buf, depth = [], [], 0
    in_string = escaped = started = False
    for ch in text:
        if not started:
            if ch == "{":
                started, depth, buf = True, 1, ["{"]
            continue
        buf.append(ch)
        if in_string:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    s = "".join(buf)
                    started, buf = False, []
                    try:
                        objs.append(json.loads(s))
                    except Exception:
                        pass
    return _normalize_records(objs)

base_rows = load_qa_records(base_path)
seg_rows  = load_qa_records(seg_path)

print("\n================ FILES / RAW COUNTS ================")
print(f"BASE: {base_path} | N={len(base_rows):,}")
print(f"SEG : {seg_path}  | N={len(seg_rows):,}")

# ---------------------------
# 4. EXACT strict-fair normalization/pairing
# ---------------------------
_punct_space_left  = re.compile(r"\s+([.,!?;:%)\]\}])")
_punct_space_right = re.compile(r"([(\[\{])\s+")
_multi_space       = re.compile(r"\s+")


def _norm_space_punct(t: str) -> str:
    t = t.replace(" - ", "-")
    t = _punct_space_left.sub(r"\1", t)
    t = _punct_space_right.sub(r"\1", t)
    t = _multi_space.sub(" ", t).strip()
    return t


def morph_marker_view(text: str) -> str:
    return _norm_space_punct("" if text is None else str(text))


def clean_view(text: str) -> str:
    t = "" if text is None else str(text)
    t = t.replace("@@ ", "").replace("@@", "")
    return _norm_space_punct(t)


def norm_for_exact(text: str) -> str:
    return re.sub(r"\s+", " ", _norm_space_punct(str(text)).lower()).strip()


def tokens(text: str) -> List[str]:
    t = _norm_space_punct(str(text)).lower()
    return re.findall(r"[a-zA-Zа-яА-ЯәғқңөұүһіӘҒҚҢӨҰҮҺІ0-9]+", t)


def token_f1(pred: str, gold: str) -> float:
    from collections import Counter
    p, g = tokens(pred), tokens(gold)
    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0
    pc, gc = Counter(p), Counter(g)
    inter = sum((pc & gc).values())
    if inter == 0:
        return 0.0
    precision = inter / len(p)
    recall    = inter / len(g)
    return float(2 * precision * recall / (precision + recall + 1e-12))


def first_by_clean_key(rows: List[Dict[str, str]]) -> Dict[str, Dict[str, str]]:
    m = {}
    for r in rows:
        k = clean_view(r["question"])
        if k and k not in m:
            m[k] = r
    return m

base_map = first_by_clean_key(base_rows)
seg_map  = first_by_clean_key(seg_rows)
common_keys = sorted(set(base_map) & set(seg_map))

paired = []
for pair_id, k in enumerate(common_keys):
    b, s = base_map[k], seg_map[k]
    paired.append({
        "pair_id": pair_id,
        "pair_key": k,
        "base_q": clean_view(b["question"]),
        "seg_q_clean": clean_view(s["question"]),
        "seg_q_morph": morph_marker_view(s["question"]),
        # STRICT FAIR: same clean baseline answer text for both conditions
        "base_a": _norm_space_punct(b["answer"]),
    })

print("\n================ STRICT PAIRING AUDIT ================")
print(f"Clean records                = {len(base_rows):,}")
print(f"Segmented records            = {len(seg_rows):,}")
print(f"Clean unique keys            = {len(base_map):,}")
print(f"Segmented unique keys        = {len(seg_map):,}")
print(f"Strict-fair paired keys      = {len(paired):,}")
print(f"Clean duplicates collapsed   = {len(base_rows)-len(base_map):,}")
print(f"Segmented duplicates collapsed = {len(seg_rows)-len(seg_map):,}")

EXPECTED = {
    "base_records": 14991,
    "seg_records": 14998,
    "base_unique": 14689,
    "seg_unique": 14696,
    "paired": 6759,
}
ACTUAL = {
    "base_records": len(base_rows),
    "seg_records": len(seg_rows),
    "base_unique": len(base_map),
    "seg_unique": len(seg_map),
    "paired": len(paired),
}
wrong = {k: (ACTUAL[k], v) for k, v in EXPECTED.items() if ACTUAL[k] != v}
if wrong:
    raise RuntimeError(f"STOP: strict-fair source counts differ from manuscript: {wrong}")
print("✅ Pairing counts exactly reproduce the strict-fair corpus.")

# Same split logic as non-FT strict-fair code
train_rows, test_rows = train_test_split(
    paired,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)

if len(train_rows) != 6083 or len(test_rows) != 676:
    raise RuntimeError(f"STOP: expected 6083 train / 676 test, got {len(train_rows)} / {len(test_rows)}")

# Split fingerprint and manifest for exact reuse in BGE/E5
train_keys = sorted(x["pair_key"] for x in train_rows)
test_keys  = sorted(x["pair_key"] for x in test_rows)
split_hash = hashlib.sha256(("\n".join(train_keys) + "\n---TEST---\n" + "\n".join(test_keys)).encode("utf-8")).hexdigest()

# Hard guard: this MUST be the identical split already used by the
# MiniLM strict-fair FT run. If it differs, cross-model comparison stops.
EXPECTED_SPLIT_SHA256 = "867d3305fbc71068afedeae71b4ef10221968d23270202bf0eda860fc8bf880b"
if split_hash != EXPECTED_SPLIT_SHA256:
    raise RuntimeError(
        "STOP: E5 strict-fair split does not match the MiniLM/BGE-M3 split. "
        f"Got {split_hash}, expected {EXPECTED_SPLIT_SHA256}."
    )

split_manifest = []
train_key_set = set(train_keys)
for r in paired:
    split_manifest.append({
        "pair_id": r["pair_id"],
        "pair_key": r["pair_key"],
        "split": "train" if r["pair_key"] in train_key_set else "test",
    })
pd.DataFrame(split_manifest).to_csv(OUT_DIR / "strict_fair_split_seed42.csv", index=False, encoding="utf-8-sig")

print("\n================ ONE STRICT-FAIR SPLIT ================")
print(f"Total paired = {len(paired):,}")
print(f"Train        = {len(train_rows):,}")
print(f"Test         = {len(test_rows):,}")
print(f"Seed         = {SEED}")
print(f"Split SHA256 = {split_hash}")
print("Gold / candidate answers = CLEAN baseline answers in BOTH conditions ✅")

# ---------------------------
# 5. E5 input formatting + retrieval/evaluation helpers
# ---------------------------
# multilingual-e5-large-instruct requires an instruction on the QUERY side.
# Passage/answer texts remain plain. This convention is applied consistently
# in non-FT evaluation, FT training, and post-FT evaluation.
E5_QUERY_INSTRUCTION = (
    "Instruct: Given a question in Kazakh, "
    "retrieve the most relevant answer.\nQuery: "
)

def e5_wrap_query(text: str) -> str:
    return E5_QUERY_INSTRUCTION + str(text)

def encode_e5_query(model: SentenceTransformer, texts: List[str], batch_size: int = ENCODE_BATCH) -> np.ndarray:
    return model.encode(
        [e5_wrap_query(t) for t in texts],
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

def encode_e5_passage(model: SentenceTransformer, texts: List[str], batch_size: int = ENCODE_BATCH) -> np.ndarray:
    return model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

@dataclass
class QAIndex:
    condition: str
    q_text: List[str]
    q_emb: np.ndarray
    ans_text: List[str]
    a_emb: np.ndarray


def build_index(model: SentenceTransformer, rows: List[Dict[str, str]], condition: str) -> QAIndex:
    if condition == "BASE":
        q_text = [r["base_q"] for r in rows]
    elif condition == "SEG":
        q_text = [r["seg_q_morph"] for r in rows]
    else:
        raise ValueError("condition must be BASE or SEG")

    ans_text = [r["base_a"] for r in rows]
    q_emb = encode_e5_query(model, q_text, batch_size=ENCODE_BATCH)
    a_emb = encode_e5_passage(model, ans_text, batch_size=ENCODE_BATCH)
    return QAIndex(condition, q_text, q_emb, ans_text, a_emb)


def eval_condition(model: SentenceTransformer, train_rows, test_rows, condition: str, stage: str):
    idx = build_index(model, train_rows, condition)

    if condition == "BASE":
        test_q = [r["base_q"] for r in test_rows]
    else:
        test_q = [r["seg_q_morph"] for r in test_rows]

    test_q_emb = encode_e5_query(model, test_q, batch_size=ENCODE_BATCH)
    gold = [r["base_a"] for r in test_rows]
    gold_a_emb = encode_e5_passage(model, gold, batch_size=ENCODE_BATCH)

    details = []
    for i, r in enumerate(test_rows):
        sims = np.dot(idx.q_emb, test_q_emb[i])
        j = int(np.argmax(sims))
        qsim = float(sims[j])
        pred = idx.ans_text[j]
        g = gold[i]
        exact = float(norm_for_exact(pred) == norm_for_exact(g))
        tf1 = token_f1(pred, g)
        ans_cos = float(np.dot(idx.a_emb[j], gold_a_emb[i]))
        semhit = float(ans_cos >= SEM_THR)
        details.append({
            "stage": stage,
            "condition": condition,
            "pair_id": r["pair_id"],
            "pair_key": r["pair_key"],
            "test_question": test_q[i],
            "gold_answer": g,
            "pred_answer": pred,
            "retrieved_train_question": idx.q_text[j],
            "retrieved_train_row": j,
            "QSim": qsim,
            "Exact": exact,
            "TokenF1": tf1,
            "AnsCos": ans_cos,
            "SemHit": semhit,
        })
    return details


def verify_paired_details(base_det, seg_det):
    if len(base_det) != len(seg_det) or len(base_det) != 676:
        raise ValueError("Paired detail lengths are not 676/676.")
    for i, (b, s) in enumerate(zip(base_det, seg_det)):
        if b["pair_key"] != s["pair_key"]:
            raise ValueError(f"Pair-key mismatch at item {i}")
        if norm_for_exact(b["gold_answer"]) != norm_for_exact(s["gold_answer"]):
            raise ValueError(f"Gold-answer mismatch at item {i}")

# ---------------------------
# 6. BERTScore item-level, common language
# ---------------------------
def add_bertscore_pair(base_det, seg_det, preferred_lang: Optional[str] = None) -> str:
    verify_paired_details(base_det, seg_det)
    if not RUN_BERTSCORE:
        for r in base_det + seg_det:
            r["BERTScoreF1"] = np.nan
        return "disabled"

    bp = [r["pred_answer"] for r in base_det]
    sp = [r["pred_answer"] for r in seg_det]
    gg = [r["gold_answer"] for r in base_det]

    langs = [preferred_lang] if preferred_lang else ["kk", "tr", "en"]
    errors = {}
    for lang in langs:
        if not lang:
            continue
        try:
            _, _, fb = bert_score(bp, gg, lang=lang, rescale_with_baseline=True,
                                  batch_size=BERTSCORE_BATCH, verbose=False)
            _, _, fs = bert_score(sp, gg, lang=lang, rescale_with_baseline=True,
                                  batch_size=BERTSCORE_BATCH, verbose=False)
            ab = fb.detach().cpu().numpy().astype(float)
            a_s = fs.detach().cpu().numpy().astype(float)
            if len(ab) != 676 or len(a_s) != 676:
                raise ValueError("Unexpected BERTScore output length")
            for i in range(676):
                base_det[i]["BERTScoreF1"] = float(ab[i])
                seg_det[i]["BERTScoreF1"]  = float(a_s[i])
            return lang
        except Exception as e:
            errors[lang] = repr(e)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    raise RuntimeError(f"BERTScore failed for all tried languages: {errors}")

# ---------------------------
# 7. Summary metrics
# ---------------------------
def metrics_from_details(details: List[Dict[str, Any]]) -> Dict[str, float]:
    return {
        "Exact@1": float(np.mean([r["Exact"] for r in details])),
        "TokenF1@1": float(np.mean([r["TokenF1"] for r in details])),
        "MeanCos@1(QSim)": float(np.mean([r["QSim"] for r in details])),
        f"Semantic@1(ans_cos≥{SEM_THR})": float(np.mean([r["SemHit"] for r in details])),
        "BERTScoreF1@1": float(np.nanmean([r.get("BERTScoreF1", np.nan) for r in details])),
    }


def summary_row(stage: str, condition: str, details: List[Dict[str, Any]]) -> Dict[str, Any]:
    row = {"Stage": stage, "Condition": condition, "N": len(details)}
    row.update(metrics_from_details(details))
    return row

# ---------------------------
# 8. FT model training — strict-fair
# ---------------------------
def protocol_metadata(condition: str) -> Dict[str, Any]:
    return {
        "model_name": MODEL_NAME,
        "condition": condition,
        "paired_total": len(paired),
        "train_n": len(train_rows),
        "test_n": len(test_rows),
        "seed": SEED,
        "test_size": TEST_SIZE,
        "split_sha256": split_hash,
        "epochs": FINETUNE_EPOCHS,
        "batch": FINETUNE_BATCH,
        "lr": FINETUNE_LR,
        "warmup_fraction": FINETUNE_WARMUP_FR,
        "loss": "CachedMultipleNegativesRankingLoss (MNRL objective)",
        "effective_batch": FINETUNE_BATCH,
        "cache_mini_batch": FINETUNE_MINI_BATCH,
        "optimizer": "bitsandbytes.PagedAdamW8bit",
        "train_answer_view": "base_a_clean",
        "train_question_view": "base_q_clean" if condition == "BASE" else "seg_q_morph",
        "query_format": E5_QUERY_INSTRUCTION,
        "answer_format": "plain clean answer text",
    }


def load_or_train(condition: str, save_dir: str) -> SentenceTransformer:
    save_path = Path(save_dir)
    meta_path = save_path / "strictfair_protocol.json"
    expected_meta = protocol_metadata(condition)

    if REUSE_SAVED_MODELS and save_path.exists() and meta_path.exists():
        existing = json.loads(meta_path.read_text(encoding="utf-8"))
        if existing == expected_meta:
            print(f"\n✅ Reusing verified saved model: {save_dir}")
            return SentenceTransformer(str(save_path), device=DEVICE)
        raise RuntimeError(
            f"Saved model directory exists but protocol metadata does not match: {save_dir}. "
            "Rename/delete that directory or set a new SAVE_DIR."
        )

    # Important: reset same seeds before EACH condition so both start from
    # the same pretrained checkpoint and use the same seeded batch ordering.
    set_all_seeds(SEED)
    model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    # E5-large-instruct checkpoints can expose trainable FP16 parameters; converting
    # trainable weights to FP32 avoids AMP GradScaler unscale errors.
    model = model.float()

    samples = []
    for r in train_rows:
        q_raw = r["base_q"] if condition == "BASE" else r["seg_q_morph"]
        a = r["base_a"]  # SAME CLEAN answer text in both FT conditions
        if q_raw and a:
            q = e5_wrap_query(q_raw)
            samples.append(InputExample(texts=[q, a]))

    if len(samples) != 6083:
        raise RuntimeError(f"Expected exactly 6083 FT pairs, got {len(samples)}")

    generator = torch.Generator()
    generator.manual_seed(SEED)
    loader = DataLoader(
        samples,
        shuffle=True,
        batch_size=FINETUNE_BATCH,
        drop_last=True,
        generator=generator,
    )
    if not hasattr(losses, "CachedMultipleNegativesRankingLoss"):
        raise RuntimeError(
            "CachedMultipleNegativesRankingLoss is unavailable. "
            "Please upgrade sentence-transformers."
        )

    # E5-large-instruct is substantially larger than MiniLM. The cached variant keeps
    # the same MNRL in-batch-negative objective over an EFFECTIVE batch of 32,
    # while recomputing gradients in mini-batches of 2 to fit a T4 GPU.
    loss = losses.CachedMultipleNegativesRankingLoss(
        model,
        mini_batch_size=FINETUNE_MINI_BATCH,
    )
    steps_per_epoch = len(loader)
    total_steps = steps_per_epoch * FINETUNE_EPOCHS
    warmup_steps = int(total_steps * FINETUNE_WARMUP_FR)

    print("\n" + "=" * 72)
    print(f"FINE-TUNING CONDITION: {condition}")
    print("=" * 72)
    print(f"Start checkpoint    : {MODEL_NAME}")
    print(f"Train pairs         : {len(samples):,}")
    print(f"Question view       : {'CLEAN' if condition == 'BASE' else 'MORPH (@@ kept)'} + E5 query instruction")
    print("Answer view         : CLEAN baseline answer, plain text (same in both conditions)")
    print(f"Epochs              : {FINETUNE_EPOCHS}")
    print(f"Effective batch     : {FINETUNE_BATCH}")
    print(f"Cache mini-batch    : {FINETUNE_MINI_BATCH}")
    print(f"Steps/epoch         : {steps_per_epoch}")
    print(f"Total optimizer steps (approx): {total_steps}")
    print(f"Learning rate       : {FINETUNE_LR}")
    print(f"Warmup steps        : {warmup_steps} ({FINETUNE_WARMUP_FR:.0%})")
    print("Loss                 : CachedMultipleNegativesRankingLoss (MNRL objective)")
    print("Optimizer            : bitsandbytes PagedAdamW8bit")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            free_b, total_b = torch.cuda.mem_get_info()
            print(f"GPU memory before fit: free={free_b/1024**3:.2f} GiB / total={total_b/1024**3:.2f} GiB")
        except Exception:
            pass

    t0 = time.time()
    model.fit(
        train_objectives=[(loader, loss)],
        epochs=FINETUNE_EPOCHS,
        warmup_steps=warmup_steps,
        optimizer_class=bnb.optim.PagedAdamW8bit,
        optimizer_params={"lr": FINETUNE_LR},
        show_progress_bar=True,
        use_amp=torch.cuda.is_available(),
        checkpoint_path=None,
    )
    elapsed = time.time() - t0

    save_path.mkdir(parents=True, exist_ok=True)
    model.save(str(save_path))
    meta_path.write_text(json.dumps(expected_meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"✅ Saved: {save_dir}")
    print(f"Training time: {elapsed/60:.2f} min")
    return model

# ---------------------------
# 9. Statistical inference
# ---------------------------
def _paired_arrays(a, b):
    x = np.asarray(a, dtype=float).reshape(-1)
    y = np.asarray(b, dtype=float).reshape(-1)
    if x.shape != y.shape or x.size == 0:
        raise ValueError("Invalid paired arrays")
    if not np.all(np.isfinite(x)) or not np.all(np.isfinite(y)):
        raise ValueError("NaN/Inf in paired arrays")
    return x, y


def paired_bootstrap_ci(base_vals, seg_vals, n_boot=N_BOOT, alpha=STAT_ALPHA, seed=STAT_SEED):
    b, s = _paired_arrays(base_vals, seg_vals)
    d = s - b
    obs = float(np.mean(d))
    rng = np.random.default_rng(seed)
    n = len(d)
    boots = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[i] = float(np.mean(d[idx]))
    lo, hi = np.quantile(boots, [alpha/2, 1-alpha/2])
    return obs, float(lo), float(hi)


def paired_signflip_p(base_vals, seg_vals, n_perm=N_PERM, seed=STAT_SEED):
    b, s = _paired_arrays(base_vals, seg_vals)
    d = s - b
    if np.allclose(d, 0):
        return 1.0
    obs = abs(float(np.mean(d)))
    rng = np.random.default_rng(seed)
    extreme = 0
    for _ in range(n_perm):
        signs = rng.choice(np.array([-1.0, 1.0]), size=len(d))
        stat = abs(float(np.mean(d * signs)))
        if stat >= obs - 1e-15:
            extreme += 1
    return float((extreme + 1) / (n_perm + 1))


def exact_mcnemar(base_vals, seg_vals):
    b = np.asarray(base_vals, dtype=int)
    s = np.asarray(seg_vals, dtype=int)
    if b.shape != s.shape:
        raise ValueError("McNemar shape mismatch")
    n10 = int(np.sum((b == 1) & (s == 0)))
    n01 = int(np.sum((b == 0) & (s == 1)))
    n = n10 + n01
    if n == 0:
        return n10, n01, 1.0
    p = binomtest(n10, n=n, p=0.5, alternative="two-sided").pvalue
    return n10, n01, float(p)


def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adjusted_sorted = np.empty(m, dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(order):
        candidate = (m - rank) * p[idx]
        running_max = max(running_max, candidate)
        adjusted_sorted[rank] = min(1.0, running_max)
    adjusted = np.empty(m, dtype=float)
    for rank, idx in enumerate(order):
        adjusted[idx] = adjusted_sorted[rank]
    return adjusted


def run_ft_paired_stats(base_det, seg_det) -> pd.DataFrame:
    verify_paired_details(base_det, seg_det)
    specs = [
        ("Exact@1", "Exact", "binary"),
        ("TokenF1@1", "TokenF1", "continuous"),
        ("MeanCos@1(QSim)", "QSim", "continuous"),
        (f"Semantic@1(ans_cos≥{SEM_THR})", "SemHit", "binary"),
        ("BERTScoreF1@1", "BERTScoreF1", "continuous"),
    ]
    rows = []
    for j, (metric, key, typ) in enumerate(specs, start=1):
        b = np.asarray([r[key] for r in base_det], dtype=float)
        s = np.asarray([r[key] for r in seg_det], dtype=float)
        delta, lo, hi = paired_bootstrap_ci(b, s, seed=STAT_SEED + j)
        n10 = n01 = np.nan
        if typ == "binary":
            n10, n01, p = exact_mcnemar(b.astype(int), s.astype(int))
            test = "Exact two-sided McNemar"
        else:
            p = paired_signflip_p(b, s, seed=STAT_SEED + 1000 + j)
            test = "Two-sided paired sign-flip permutation"
        rows.append({
            "Model": MODEL_NAME,
            "Stage": "FINE_TUNED_STRICT_FAIR",
            "Comparison": "FT_BASE_CLEAN vs FT_SEG_MORPH_ONLY",
            "Metric": metric,
            "N": len(b),
            "Baseline": float(np.mean(b)),
            "Segmented": float(np.mean(s)),
            "Delta_SegMinusBase": delta,
            "CI95_low": lo,
            "CI95_high": hi,
            "p_raw": p,
            "Test": test,
            "n10_Base1_Seg0": n10,
            "n01_Base0_Seg1": n01,
        })
    df = pd.DataFrame(rows)
    df["p_Holm"] = holm_adjust(df["p_raw"].to_numpy())
    df["Significant_Holm"] = df["p_Holm"] < STAT_ALPHA
    df["Direction"] = np.where(df["Delta_SegMinusBase"] > 0, "Segmented higher",
                        np.where(df["Delta_SegMinusBase"] < 0, "Segmented lower", "No difference"))
    df["CI95"] = df.apply(lambda r: f"[{r.CI95_low:.6f}, {r.CI95_high:.6f}]", axis=1)
    return df

# ---------------------------
# 10. Run NON-FT reference
# ---------------------------
all_summary_rows = []
nonft_base_det = nonft_seg_det = None
common_bert_lang = None

if RUN_NONFT_REFERENCE:
    print("\n" + "=" * 72)
    print("NON-FINE-TUNED STRICT-FAIR REFERENCE")
    print("=" * 72)
    set_all_seeds(SEED)
    nonft_model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    nonft_base_det = eval_condition(nonft_model, train_rows, test_rows, "BASE", "NON_FT")
    nonft_seg_det  = eval_condition(nonft_model, train_rows, test_rows, "SEG", "NON_FT")
    verify_paired_details(nonft_base_det, nonft_seg_det)
    if RUN_BERTSCORE:
        common_bert_lang = add_bertscore_pair(nonft_base_det, nonft_seg_det)
        print(f"BERTScore language used: {common_bert_lang}")
    all_summary_rows += [
        summary_row("NON_FT", "BASE_CLEAN", nonft_base_det),
        summary_row("NON_FT", "SEG_MORPH_ONLY", nonft_seg_det),
    ]
    pd.DataFrame(nonft_base_det).to_csv(OUT_DIR / "nonft_base_details.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(nonft_seg_det).to_csv(OUT_DIR / "nonft_seg_details.csv", index=False, encoding="utf-8-sig")
    del nonft_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ---------------------------
# 11. Train/evaluate FT BASE
# ---------------------------
base_model = load_or_train("BASE", BASE_SAVE_DIR)
ft_base_det = eval_condition(base_model, train_rows, test_rows, "BASE", "FT")
pd.DataFrame(ft_base_det).to_csv(OUT_DIR / "ft_base_details_pre_bertscore.csv", index=False, encoding="utf-8-sig")
del base_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ---------------------------
# 12. Train/evaluate FT SEG
# ---------------------------
seg_model = load_or_train("SEG", SEG_SAVE_DIR)
ft_seg_det = eval_condition(seg_model, train_rows, test_rows, "SEG", "FT")
pd.DataFrame(ft_seg_det).to_csv(OUT_DIR / "ft_seg_details_pre_bertscore.csv", index=False, encoding="utf-8-sig")
del seg_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

verify_paired_details(ft_base_det, ft_seg_det)

# BERTScore AFTER neural models are released from GPU as much as possible
if RUN_BERTSCORE:
    ft_bert_lang = add_bertscore_pair(ft_base_det, ft_seg_det, preferred_lang=common_bert_lang)
    print(f"\nFT BERTScore language used: {ft_bert_lang}")
else:
    ft_bert_lang = "disabled"

pd.DataFrame(ft_base_det).to_csv(OUT_DIR / "ft_base_details.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(ft_seg_det).to_csv(OUT_DIR / "ft_seg_details.csv", index=False, encoding="utf-8-sig")

all_summary_rows += [
    summary_row("FT", "BASE_CLEAN", ft_base_det),
    summary_row("FT", "SEG_MORPH_ONLY", ft_seg_det),
]

summary_df = pd.DataFrame(all_summary_rows)
summary_df.to_csv(OUT_DIR / "e5_large_strictfair_nonft_and_ft_summary.csv", index=False, encoding="utf-8-sig")

# ---------------------------
# 13. Primary FT paired statistical inference
# ---------------------------
stats_df = run_ft_paired_stats(ft_base_det, ft_seg_det)
stats_df["BERTScore_lang"] = ft_bert_lang
stats_df.to_csv(OUT_DIR / "e5_large_strictfair_ft_paired_statistics.csv", index=False, encoding="utf-8-sig")

# ---------------------------
# 14. Comparison FT vs NON-FT
# ---------------------------
metric_labels = [
    "Exact@1", "TokenF1@1", "MeanCos@1(QSim)",
    f"Semantic@1(ans_cos≥{SEM_THR})", "BERTScoreF1@1"
]

comparison_rows = []
if RUN_NONFT_REFERENCE:
    nb = metrics_from_details(nonft_base_det)
    ns = metrics_from_details(nonft_seg_det)
    fb = metrics_from_details(ft_base_det)
    fs = metrics_from_details(ft_seg_det)
    for m in metric_labels:
        comparison_rows.append({
            "Metric": m,
            "NonFT_BASE": nb[m],
            "NonFT_SEG": ns[m],
            "NonFT_Delta_SEGminusBASE": ns[m] - nb[m],
            "FT_BASE": fb[m],
            "FT_SEG": fs[m],
            "FT_Delta_SEGminusBASE": fs[m] - fb[m],
            "BASE_FTminusNonFT": fb[m] - nb[m],
            "SEG_FTminusNonFT": fs[m] - ns[m],
        })
comparison_df = pd.DataFrame(comparison_rows)
if not comparison_df.empty:
    comparison_df.to_csv(OUT_DIR / "e5_large_strictfair_ft_vs_nonft_comparison.csv", index=False, encoding="utf-8-sig")

# ---------------------------
# 15. Protocol metadata
# ---------------------------
protocol = {
    "model": MODEL_NAME,
    "device": DEVICE,
    "raw_clean_records": len(base_rows),
    "raw_segmented_records": len(seg_rows),
    "clean_unique_keys": len(base_map),
    "seg_unique_keys": len(seg_map),
    "paired_total": len(paired),
    "train_n": len(train_rows),
    "test_n": len(test_rows),
    "seed": SEED,
    "test_size": TEST_SIZE,
    "split_sha256": split_hash,
    "base_ft": "clean question -> clean baseline answer",
    "seg_ft": "morph-marked segmented question -> SAME clean baseline answer",
    "e5_query_instruction": E5_QUERY_INSTRUCTION,
    "answer_encoding": "plain text / no query instruction",
    "loss": "CachedMultipleNegativesRankingLoss (MNRL objective)",
    "optimizer": "bitsandbytes.PagedAdamW8bit",
    "epochs": FINETUNE_EPOCHS,
    "effective_batch_size": FINETUNE_BATCH,
    "cache_mini_batch_size": FINETUNE_MINI_BATCH,
    "learning_rate": FINETUNE_LR,
    "warmup_fraction": FINETUNE_WARMUP_FR,
    "semantic_threshold": SEM_THR,
    "bootstrap_n": N_BOOT,
    "permutation_n": N_PERM,
    "holm_family": metric_labels,
    "bertscore_language": ft_bert_lang,
}
(OUT_DIR / "protocol.json").write_text(json.dumps(protocol, ensure_ascii=False, indent=2), encoding="utf-8")

# ---------------------------
# 16. Print article-ready results
# ---------------------------
print("\n\n" + "=" * 72)
print("TABLE 1. E5-LARGE-INSTRUCT STRICT-FAIR: NON-FT AND FT RESULTS")
print("=" * 72)
print(summary_df.to_string(index=False))

if not comparison_df.empty:
    print("\n\n" + "=" * 72)
    print("TABLE 2. DIRECT COMPARISON WITH NON-FINE-TUNED RESULTS")
    print("=" * 72)
    print(comparison_df.to_string(index=False))

print("\n\n" + "=" * 72)
print("TABLE 3. PRIMARY PAIRED INFERENCE: FT SEGMENTED - FT BASELINE")
print("=" * 72)
display_cols = [
    "Metric", "N", "Baseline", "Segmented", "Delta_SegMinusBase", "CI95",
    "p_raw", "p_Holm", "Significant_Holm", "Direction"
]
print(stats_df[display_cols].to_string(index=False))

print("\nMcNemar discordant-pair counts:")
print(stats_df[stats_df["Metric"].isin(["Exact@1", f"Semantic@1(ans_cos≥{SEM_THR})"])][
    ["Metric", "n10_Base1_Seg0", "n01_Base0_Seg1", "p_raw", "p_Holm"]
].to_string(index=False))

# QSim-specific block for reviewer discussion
if RUN_NONFT_REFERENCE:
    q = comparison_df[comparison_df["Metric"] == "MeanCos@1(QSim)"].iloc[0]
    print("\n\n" + "=" * 72)
    print("QSIM / FINE-TUNING SPECIALIZATION CHECK")
    print("=" * 72)
    print(f"Non-FT BASE QSim = {q['NonFT_BASE']:.6f}")
    print(f"FT BASE QSim     = {q['FT_BASE']:.6f}")
    print(f"BASE FT-NONFT    = {q['BASE_FTminusNonFT']:+.6f}")
    print(f"Non-FT SEG QSim  = {q['NonFT_SEG']:.6f}")
    print(f"FT SEG QSim      = {q['FT_SEG']:.6f}")
    print(f"SEG FT-NONFT     = {q['SEG_FTminusNonFT']:+.6f}")

# ---------------------------
# 17. Zip outputs
# ---------------------------
zip_path = Path("e5_large_strictfair_problem2_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.glob("*"):
        if p.is_file():
            z.write(p, arcname=p.name)

print("\n\n" + "=" * 72)
print("FILES SAVED")
print("=" * 72)
for p in sorted(OUT_DIR.glob("*")):
    print(p)
print(zip_path)

# ---------------------------
# 18. Compact block to send back to ChatGPT
# ---------------------------
print("\n\n" + "=" * 72)
print("COPY THIS BLOCK BACK TO CHATGPT")
print("=" * 72)
print(f"Model = {MODEL_NAME}")
print(f"Paired total = {len(paired)}")
print(f"Train = {len(train_rows)}")
print(f"Test = {len(test_rows)}")
print(f"Seed = {SEED}")
print(f"Split SHA256 = {split_hash}")
print(f"FT protocol BASE = clean question -> clean answer")
print(f"FT protocol SEG  = morph question -> SAME clean answer")
print("E5 query format = Instruct: Given a question in Kazakh, retrieve the most relevant answer.\nQuery: <question>")
print("E5 answer format = plain clean answer text")
print(f"Epochs={FINETUNE_EPOCHS}; effective_batch={FINETUNE_BATCH}; cache_mini_batch={FINETUNE_MINI_BATCH}; lr={FINETUNE_LR}; warmup={FINETUNE_WARMUP_FR:.0%}; loss=CachedMNRL; optimizer=PagedAdamW8bit")
print(f"BERTScore language = {ft_bert_lang}")

print("\nSUMMARY:")
print(summary_df.to_string(index=False))

if not comparison_df.empty:
    print("\nFT vs NON-FT:")
    print(comparison_df.to_string(index=False))

print("\nFT PAIRED STATISTICS:")
print(stats_df[display_cols].to_string(index=False))
print("\n✅ DONE")

# Optional Colab download:
# if IN_COLAB:
#     files.download(str(zip_path))


/tmp/ipykernel_2282/2150131548.py:76: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


E5-LARGE-INSTRUCT STRICT-FAIR FINE-TUNING — REVIEWER 1 / PROBLEM 2
Device: cuda
GPU: Tesla T4
Model: intfloat/multilingual-e5-large-instruct
E5 input formatting: instruction-wrapped QUERY side; plain PASSAGE/answer side
Effective FT batch=32; CachedMNRL mini-batch=2

================ FILES / RAW COUNTS ================
BASE: /content/baseline_15000.json | N=14,991
SEG : /content/kazakh_segmented_15000.json  | N=14,998

================ STRICT PAIRING AUDIT ================
Clean records                = 14,991
Segmented records            = 14,998
Clean unique keys            = 14,689
Segmented unique keys        = 14,696
Strict-fair paired keys      = 6,759
Clean duplicates collapsed   = 302
Segmented duplicates collapsed = 302
✅ Pairing counts exactly reproduce the strict-fair corpus.

================ ONE STRICT-FAIR SPLIT ================
Total paired = 6,759
Train        = 6,083
Test         = 676
Seed         = 42
Split SHA256 = 867d3305fbc71068afedeae71b4ef10221968d23270202bf0ed

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/140k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



FT BERTScore language used: kk


TABLE 1. E5-LARGE-INSTRUCT STRICT-FAIR: NON-FT AND FT RESULTS
 Stage      Condition   N  Exact@1  TokenF1@1  MeanCos@1(QSim)  Semantic@1(ans_cos≥0.85)  BERTScoreF1@1
NON_FT     BASE_CLEAN 676 0.002959   0.400414         0.960814                  0.991124       0.810350
NON_FT SEG_MORPH_ONLY 676 0.004438   0.367864         0.967879                  0.982249       0.800921
    FT     BASE_CLEAN 676 0.002959   0.425264         0.776877                  0.218935       0.816041
    FT SEG_MORPH_ONLY 676 0.002959   0.420949         0.776625                  0.202663       0.814233


TABLE 2. DIRECT COMPARISON WITH NON-FINE-TUNED RESULTS
                  Metric  NonFT_BASE  NonFT_SEG  NonFT_Delta_SEGminusBASE  FT_BASE   FT_SEG  FT_Delta_SEGminusBASE  BASE_FTminusNonFT  SEG_FTminusNonFT
                 Exact@1    0.002959   0.004438                  0.001479 0.002959 0.002959               0.000000           0.000000         -0.001479
               TokenF1@